In [2]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import broadcast
import logging

In [3]:
##%%
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:2.7.3") \
    .appName("spark-hw") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e3b75705-1f67-44c9-8ea2-69108234f3ad;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;2.7.3 in central
	found org.apache.hadoop#hadoop-common;2.7.3 in central
	found org.apache.hadoop#hadoop-annotations;2.7.3 in central
	found com.google.guava#guava;11.0.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found commons-cli#commons-cli;1.2 in central
	found org.apache.commons#commons-math3;3.1.1 in central
	found xmlenc#xmlenc;0.52 in central
	found commons-httpclient#commons-httpclient;3.1 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.4 in central
	found commons-io#commons-io;2.4 in central
	found commons-net#commons-net;3.1 in central
	found commons-collections#commons-collections;3.2.2 in 

In [4]:
actor_df = spark.read.csv('../data/actor.csv', header=True, inferSchema=True)
address_df = spark.read.csv('../data/address.csv', header=True, inferSchema=True)
category_df = spark.read.csv('../data/category.csv', header=True, inferSchema=True)
city_df = spark.read.csv('../data/city.csv', header=True, inferSchema=True)
country_df = spark.read.csv('../data/country.csv', header=True, inferSchema=True)
customer_df = spark.read.csv('../data/customer.csv', header=True, inferSchema=True)
film_df = spark.read.csv('../data/film.csv', header=True, inferSchema=True)
film_actor_df = spark.read.csv('../data/film_actor.csv', header=True, inferSchema=True)
film_category_df = spark.read.csv('../data/film_category.csv', header=True, inferSchema=True)
inventory_df = spark.read.csv('../data/inventory.csv', header=True, inferSchema=True)
language_df = spark.read.csv('../data/language.csv', header=True, inferSchema=True)
payment_df = spark.read.csv('../data/payment.csv', header=True, inferSchema=True)
rental_df = spark.read.csv('../data/rental.csv', header=True, inferSchema=True)
staff_df = spark.read.csv('../data/staff.csv', header=True, inferSchema=True)
store_df = spark.read.csv('../data/store.csv', header=True, inferSchema=True)

# Домашнє завдання на тему Spark SQL

Задачі з домашнього завдання на SQL потрібно розвʼязати за допомогою Spark SQL DataFrame API.

- Дампи таблиць знаходяться в папці `data`. Датафрейми таблиць вже створені в клітинці вище.
- Можете створювати стільки нових клітинок, скільки вам необхідно.
- Розвʼязок кожної задачі має бути відображений в самому файлі (використати метод `.show()`)
- код має бути оформлений у відповідності із одним із стилем, показаним лектором на занятті 13.

**Увага!**
Використовувати мову запитів SQL безпосередньо забороняється, потрібно використовувати виключно DataFrame API!


1.
Вивести кількість фільмів в кожній категорії.
Результат відсортувати за спаданням.

In [5]:
category_df.printSchema()
category_df.count()
film_category_df.printSchema()
film_category_df.count()
fc = film_category_df.alias("fc")
c = category_df.alias("c")

films_per_category = (
    fc
    .join(broadcast(c), F.col("fc.category_id") == F.col("c.category_id"), "inner")
    .groupBy(F.col("c.name"))
    .agg(F.count("*").alias("cnt"))
    .orderBy(F.desc("cnt"))
)

films_per_category.show()

root
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

+-----------+---+
|       name|cnt|
+-----------+---+
|     Sports| 74|
|    Foreign| 73|
|     Family| 69|
|Documentary| 68|
|  Animation| 66|
|     Action| 64|
|        New| 63|
|      Drama| 62|
|      Games| 61|
|     Sci-Fi| 61|
|   Children| 60|
|     Comedy| 58|
|     Travel| 57|
|   Classics| 57|
|     Horror| 56|
|      Music| 51|
+-----------+---+



2.
Вивести 10 акторів, чиї фільми брали на прокат найбільше.
Результат відсортувати за спаданням.

In [13]:
rental_df.printSchema()
inventory_df.printSchema()
film_actor_df.printSchema()
actor_df.printSchema()
top_actors_df = (
    rental_df
    .join(inventory_df,"inventory_id","inner")
    .join(film_actor_df,"film_id", "inner")
    .join(actor_df,"actor_id", "inner")
)
top_actors_df_cols = top_actors_df.select(
    actor_df.actor_id,
    actor_df.first_name,
    actor_df.last_name
)
top_actors_by_rental = (
    top_actors_df_cols
    .groupBy("actor_id", "first_name", "last_name")
    .agg(F.count("*").alias("rentals_number"))
    .orderBy(F.desc("rentals_number"))
    .limit(10)
)
top_actors_by_rental.show()

root
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- inventory_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- actor_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- actor_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- last_update: timestamp (nullable = true)

+--------+----------+-----------+--------------+
|actor_id|first_name|  last_name|rentals_number|
+--------+----------+-----------+--------------+
|     107|      GINA|  DEGENERES|           753|
|

3.
Вивести категорія фільмів, на яку було витрачено найбільше грошей
в прокаті

In [25]:
payment_df.printSchema()
rental_df.printSchema()
inventory_df.printSchema()
film_category_df.printSchema()
category_df.printSchema()

film_categories_by_rental_df = (
    payment_df
    .join(rental_df,"rental_id","inner")
    .join(inventory_df,"inventory_id","inner")
    .join(film_category_df,"film_id","inner")
    .join(category_df,"category_id","inner")
)
film_categories_by_rental_cols = film_categories_by_rental_df.select(
        category_df.category_id,
        category_df.name,
        payment_df.amount
)
top_film_categories_by_rental = (
    film_categories_by_rental_cols
    .groupBy("category_id","name")
    .agg(F.sum("amount").alias("total_rental"))
    .withColumn("total_rental", F.round("total_rental", 2))
    .orderBy(F.desc("total_rental"))
    .limit(1)
    
)
top_film_categories_by_rental.show()

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- rental_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_date: timestamp (nullable = true)

root
 |-- rental_id: integer (nullable = true)
 |-- rental_date: timestamp (nullable = true)
 |-- inventory_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- return_date: timestamp (nullable = true)
 |-- staff_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- film_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

root
 |-- category_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- last_update: timest

DataFrame[payment_id: int, customer_id: int, staff_id: int, rental_id: int, amount: double, payment_date: timestamp]

4.
Вивести назви фільмів, яких не має в inventory.

In [34]:
film_df.printSchema()
inventory_df.printSchema()
films_not_in_inventory = film_df.join(inventory_df,"film_id","left_anti").select("title")
films_not_in_inventory.count()
films_not_in_inventory.show(films_not_in_inventory.count(),truncate=False)


root
 |-- film_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- language_id: integer (nullable = true)
 |-- original_language_id: string (nullable = true)
 |-- rental_duration: integer (nullable = true)
 |-- rental_rate: double (nullable = true)
 |-- length: integer (nullable = true)
 |-- replacement_cost: double (nullable = true)
 |-- rating: string (nullable = true)
 |-- last_update: timestamp (nullable = true)
 |-- special_features: string (nullable = true)
 |-- fulltext: string (nullable = true)

root
 |-- inventory_id: integer (nullable = true)
 |-- film_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- last_update: timestamp (nullable = true)

+----------------------+
|title                 |
+----------------------+
|ALICE FANTASIA        |
|APOLLO TEEN           |
|ARGONAUTS TOWN        |
|ARK RIDGEMONT         |
|ARSENIC INDEPENDENCE  |
|BOON

42

5.
Вивести топ 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”

In [35]:
top_children_actors = (
    film_category_df
    .join(category_df, "category_id", "inner")
    .filter(F.col("name") == "Children")
    .join(film_actor_df, "film_id", "inner")
    .join(actor_df, "actor_id", "inner")
    .groupBy("actor_id", "first_name", "last_name")
    .agg(F.count("*").alias("appearances"))
    .orderBy(F.desc("appearances"))
    .limit(3)
)

top_children_actors.show()

+--------+----------+---------+-----------+
|actor_id|first_name|last_name|appearances|
+--------+----------+---------+-----------+
|      17|     HELEN|   VOIGHT|          7|
|     127|     KEVIN|  GARLAND|          5|
|     140|    WHOOPI|     HURT|          5|
+--------+----------+---------+-----------+



Stop Spark session:

In [ ]:
spark.stop()